# Detección de daños en carretera · RDD2022 (EEUU)

En este cuaderno aprendo a preparar un dataset de detección de objetos y a entrenar un modelo **YOLO** (con Ultralytics) que encuentre daños en fotos de carreteras. Uso solo las imágenes de **Estados Unidos**.

El dataset tiene 4 tipos de daño:

| Código | Daño |
|---|---|
| D00 | Grieta longitudinal |
| D10 | Grieta transversal |
| D20 | Grieta de cocodrilo |
| D40 | Bache |

## 1. Rutas de los datos

Descomprimí `United_States.zip` dentro de `data/RDD2022/`. Uso `Path` (de `pathlib`) para guardar en variables las rutas de las carpetas:

- `TRAIN_DIR`: la carpeta de entrenamiento. Solo uso `train` porque la carpeta `test` **no tiene anotaciones**, así que no sirve ni para entrenar ni para medir.
- `IMAGES_DIR` y `XML_DIR`: las construyo a partir de `TRAIN_DIR` con la barra `/`. Si cambio de país, solo tengo que modificar `TRAIN_DIR`.

Las rutas son **relativas**: cuentan desde la carpeta donde está este cuaderno.

También defino `COLORES`, un diccionario que asigna un color fijo a cada clase. Así cada daño tendrá siempre el mismo color en todas las gráficas.

In [15]:
import os
from pathlib import Path

TRAIN_DIR = Path('data/RDD2022/United_States/train')
IMAGES_DIR = TRAIN_DIR / 'images'
XML_DIR = TRAIN_DIR / 'annotations/xmls'

# Color fijo para cada clase (lo reutilizaré en todas las gráficas del proyecto)
COLORES = {
    'D00': 'tab:blue',
    'D10': 'tab:orange',
    'D20': 'tab:red',
    'D40': 'tab:green',
}

## 2. Contar los archivos

Con el método `glob` busco los archivos que encajan con un patrón: `*.jpg` significa "cualquier nombre que termine en .jpg". `sorted` los ordena alfabéticamente para que el orden sea siempre el mismo.

Con `len` cuento cuántos hay. **Resultado:** 4.805 imágenes y 4.805 XML.

In [17]:
image_paths = list(sorted(IMAGES_DIR.glob('*.jpg')))
xml_paths = list(sorted(XML_DIR.glob('*.xml')))

len(image_paths), len(xml_paths)

(4805, 4805)

## 3. Comprobar que cada imagen tiene su XML

Que haya el mismo número de imágenes y XML no garantiza que se correspondan. Cada foto y su XML se llaman igual (solo cambia la extensión), así que:

1. Saco el nombre sin extensión de cada archivo con `.stem`.
2. Los convierto en conjuntos (`set`) y los resto: la resta me da los nombres que están en uno y no en el otro.

**Resultado:** 0 imágenes sin XML y 0 XML sin imagen. Los datos están completos.

In [20]:
image_names = [p.stem for p in image_paths]
xml_names = [p.stem for p in xml_paths]
sin_xml = set(image_names) - set(xml_names)
sin_image = set(xml_names) - set(image_names)

len(sin_xml), len(sin_image)

(0, 0)

## 4. Leer un XML

Cada XML guarda las anotaciones de una foto. Python lo ve como un **árbol**: la raíz es `<annotation>` y de ella cuelgan `<filename>`, `<size>` y un `<object>` por cada daño.

- `ET.parse` lee el archivo y `getroot()` me da la raíz.
- `findtext` me devuelve el texto de una etiqueta. Para llegar a una etiqueta que está dentro de otra uso una ruta con barras, como `size/width`.
- `findtext` siempre devuelve **texto**, así que convierto el ancho y el alto a número con `int`.

**Resultado:** `United_States_000000.jpg`, 640 × 640 píxeles.

In [25]:
import xml.etree.ElementTree as ET

xml_ejemplo = xml_paths[0]
tree = ET.parse(xml_ejemplo)
root = tree.getroot()
filename = root.findtext('filename')
width = int(root.findtext('size/width'))
height = int(root.findtext('size/height'))
print(f'filename: {filename}, width: {width}, height: {height}')


filename: United_States_000000.jpg, width: 640, height: 640


### Recorrer los daños de un XML

`findall('object')` me devuelve una **lista** con todos los `<object>` del XML. La recorro con un bucle `for`: en cada vuelta, `obj` es uno de los daños, y con `findtext` sobre `obj` saco su tipo (`name`) y su caja (`bndbox`).

Las coordenadas de la caja están en **píxeles**, contadas desde la esquina superior izquierda de la foto:
- `xmin`, `ymin`: esquina superior izquierda de la caja.
- `xmax`, `ymax`: esquina inferior derecha.

**Resultado:** este XML tiene 3 daños (dos D10 y un D00), que coinciden con el archivo.

In [31]:
objetos = root.findall('object')

for obj in objetos:
    print(f'Nombre: {obj.findtext("name")}, BBox: {obj.findtext("bndbox/xmin")}, {obj.findtext("bndbox/ymin")}, {obj.findtext("bndbox/xmax")}, {obj.findtext("bndbox/ymax")}')

Nombre: D10, BBox: 556, 510, 640, 544
Nombre: D10, BBox: 502, 458, 624, 494
Nombre: D00, BBox: 579, 552, 633, 615


## 5. Función `leer_xml`

Meto todo lo anterior en una **función** para poder leer cualquier XML en una sola línea. Recibe la ruta del XML y devuelve 4 cosas:

1. el nombre de la foto,
2. su ancho,
3. su alto,
4. una lista de daños, donde cada daño es una tupla `(clase, xmin, ymin, xmax, ymax)`.

Convierto las coordenadas con `float` y no con `int` porque en otros países del dataset tienen decimales.

In [33]:
def leer_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    filename = root.findtext('filename')
    width = int(root.findtext('size/width'))
    height = int(root.findtext('size/height'))
    
    objetos = []
    for obj in root.findall('object'):
        nombre = obj.findtext("name")
        xmin = float(obj.findtext("bndbox/xmin"))
        ymin = float(obj.findtext("bndbox/ymin"))
        xmax = float(obj.findtext("bndbox/xmax"))
        ymax = float(obj.findtext("bndbox/ymax"))
        objetos.append((nombre, xmin, ymin, xmax, ymax))
    
    return filename, width, height, objetos

## 6. Análisis de los 4.805 XML

Recorro todos los XML con `leer_xml` y voy acumulando:

- `conteo_clases`: un `Counter` que suma cuántas cajas hay de cada clase.
- `conteo_tamanos`: un `Counter` con las resoluciones de las fotos.
- `imagenes_vacias`: cuántas fotos no tienen ningún daño.
- `cajas_por_imagen`: cuántas cajas tiene cada foto, para calcular el total, el máximo y la media.

Dentro hay un bucle anidado: el de fuera recorre las fotos y el de dentro, los daños de cada foto.

In [38]:
from collections import Counter

conteo_clases = Counter()
conteo_tamanos = Counter()
imagenes_vacias = 0
cajas_por_imagen = []

for xml_path in xml_paths:
    filename, width, height, objetos = leer_xml(xml_path)
    conteo_tamanos[(width, height)] += 1
    cajas_por_imagen.append(len(objetos))
    if len(objetos) == 0:
        imagenes_vacias += 1
    for obj in objetos:
        conteo_clases[obj[0]] += 1
    
print(f'Clase más común: {conteo_clases.most_common()}')
print(f'Conteo tamaños: {conteo_tamanos}')
print(f'Número de imágenes vacías: {imagenes_vacias}')
print(f'Total de cajas: {sum(cajas_por_imagen)}')
print(f'Maximo de cajas en una imagen: {max(cajas_por_imagen)}')
print(f'Promedio de cajas por imagen: {sum(cajas_por_imagen)/len(cajas_por_imagen):.2f}')


Clase más común: [('D00', 6750), ('D10', 3295), ('D20', 834), ('D40', 135)]
Conteo tamaños: Counter({(640, 640): 4805})
Número de imágenes vacías: 0
Total de cajas: 11014
Maximo de cajas en una imagen: 11
Promedio de cajas por imagen: 2.29


### Conclusiones del análisis

| Clase | Daño | Cajas | % |
|---|---|---|---|
| D00 | Grieta longitudinal | 6.750 | 61 % |
| D10 | Grieta transversal | 3.295 | 30 % |
| D20 | Grieta de cocodrilo | 834 | 8 % |
| D40 | Bache | 135 | 1 % |

- **Solo hay 4 clases.** No hace falta filtrar etiquetas extra.
- **Hay mucho desequilibrio.** D00 aparece 50 veces más que D40. Lo más probable es que el modelo detecte mal los baches, así que tengo que mirar las métricas **por clase** y no solo la media.
- **Todas las fotos miden 640 × 640.** Es justo el tamaño por defecto de YOLO.
- **No hay fotos sin daños.** El modelo nunca verá una carretera en buen estado, así que puede dar *falsos positivos*.
- **De media hay 2,29 cajas por foto**, y como mucho 11.

## 7. Gráfica de cajas por clase

- `plt.subplots` me da la figura (`fig`, el lienzo) y los ejes (`ax`, la zona donde dibujo).
- `ax.bar` dibuja una barra por clase. Le paso una **lista de colores**, uno por barra en el mismo orden, que construyo con el diccionario `COLORES`.
- `ax.bar_label` escribe el número encima de cada barra.
- `set_title`, `set_xlabel` y `set_ylabel` ponen el título y los nombres de los ejes.

In [ ]:
import matplotlib.pyplot as plt

nombres = []
cantidades = []
colores_barras = []

for t in conteo_clases.most_common():
    nombres.append(t[0])
    cantidades.append(t[1])
    colores_barras.append(COLORES[t[0]])

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(nombres, cantidades, color=colores_barras)
ax.bar_label(ax.containers[0])
ax.set_title('Cantidad de objetos por clase')
ax.set_xlabel('Clase')
ax.set_ylabel('Cantidad')

plt.show()